### 0: Importer

In [119]:
import json
from pathlib import Path
from collections import Counter

### 1: Load and inspect dataset

In [120]:
DATA_PATH = "../out/training/epistemic_factkg_training.jsonl"   # change this path

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in open(DATA_PATH, "r", encoding="utf-8")]

print("Total records:", len(data))
print("First record keys:", data[0].keys())

Total records: 5135
First record keys: dict_keys(['schema_version', 'id', 'claim', 'verdict', 'epistemic', 'claim_triples', 'reasoning', 'evidence', 'provenance', 'meta'])


### 2: Quick schema sanity check

In [121]:
required_fields = [
    "schema_version",
    "id",
    "claim",
    "verdict",
    "epistemic",
    "evidence",
    "provenance",
    "meta"
]

def check_required_fields(record):
    missing = []
    for field in required_fields:
        if field not in record:
            missing.append(field)
    return missing

bad_records = []

for i, record in enumerate(data):
    missing = check_required_fields(record)
    if missing:
        bad_records.append((i, record.get("id", None), missing))

print("Bad records:", len(bad_records))

if bad_records[:5]:
    print(bad_records[:5])

Bad records: 0


### 3: Inspect label distribution

In [122]:
verdicts = Counter()
pramanas = Counter()
datasets = Counter()
stances = Counter()
has_claim_triples = Counter()
has_evidence_triples = Counter()

for record in data:
    verdicts[record["verdict"]["label"]] += 1
    pramanas[record["epistemic"]["pramana_primary"]] += 1
    datasets[record["provenance"]["dataset"]] += 1

    claim_triples = record.get("claim_triples")
    has_claim_triples[claim_triples is not None and len(claim_triples) > 0] += 1

    ev_has_triple = False

    for ev in record["evidence"]:
        stances[ev["stance"]] += 1

        triples = ev.get("triples")
        if triples is not None and len(triples) > 0:
            ev_has_triple = True

    has_evidence_triples[ev_has_triple] += 1


print("Verdict distribution:")
print(verdicts)

print("\nPramana distribution:")
print(pramanas)

print("\nDataset distribution:")
print(datasets)

print("\nEvidence stance distribution:")
print(stances)

print("\nHas claim triples:")
print(has_claim_triples)

print("\nHas evidence triples:")
print(has_evidence_triples)

Verdict distribution:
Counter({'refuted': 2947, 'supported': 1871, 'not_enough_evidence': 317})

Pramana distribution:
Counter({'testimony': 1813, 'perception': 1298, 'non_apprehension': 750, 'inference': 704, 'comparison_analogy': 570})

Dataset distribution:
Counter({'averitec': 3335, 'ai2thor': 1800})

Evidence stance distribution:
Counter({'refutes': 6357, 'supports': 2710, None: 936, 'absent': 750})

Has claim triples:
Counter({False: 4835, True: 300})

Has evidence triples:
Counter({False: 4085, True: 1050})


### 4: Inspect one AVeriTeC and one AI2THOR sample

In [123]:
def print_record_summary(record):
    print("ID:", record["id"])
    print("Dataset:", record["provenance"]["dataset"])
    print("Claim:", record["claim"])
    print("Verdict:", record["verdict"]["label"])
    print("Pramana:", record["epistemic"]["pramana_primary"])
    print("Weight:", record["epistemic"]["confidence_weight"])
    print("Claim triples:", record.get("claim_triples"))
    print("Reasoning:", record.get("reasoning"))

    print("\nEvidence:")
    for ev in record["evidence"]:
        print(" - Evidence ID:", ev["evidence_id"])
        print("   Text:", ev["text"])
        print("   Stance:", ev["stance"])
        print("   Modality:", ev["modality"])
        print("   Triples:", ev["triples"])
        print()


averitec_sample = next(
    r for r in data if r["provenance"]["dataset"] == "averitec"
)

ai2thor_sample = next(
    r for r in data if r["provenance"]["dataset"] == "ai2thor"
)

print("AVeriTeC sample")
print_record_summary(averitec_sample)

print("\n" + "=" * 80 + "\n")

print("AI2THOR sample")
print_record_summary(ai2thor_sample)

AVeriTeC sample
ID: averitec-train-000001
Dataset: averitec
Claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.
Verdict: supported
Pramana: testimony
Weight: 0.8
Claim triples: None
Reasoning: None

Evidence:
 - Evidence ID: averitec-train-000001-q1-a1
   Text: Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014 No
   Stance: supports
   Modality: web_text
   Triples: []

 - Evidence ID: averitec-train-000001-q2-a1
   Text: Did Hunter Biden have any experience in Ukraine at the time he joined the board of the  Burisma energy company in 2014 No
   Stance: supports
   Modality: web_text
   Triples: []



AI2THOR sample
ID: claim-FloorPlan1-onehop-sup-000000
Dataset: ai2thor
Claim: The lettuce is not dirty.
Verdict: supported
Pramana: perception
Weight: 0.95
Claim triples: [['http://epistemicfactkg.org/entities/Lettuce|-01.81|+00.97|-00.94', 'isDirty', 'Fa

### 5: Create label maps

In [124]:
verdict_map = {
    "supported": 0,
    "refuted": 1,
    "not_enough_evidence": 2,
    "conflicting_evidence": 3
}

stance_map = {
    "supports": 0,
    "refutes": 1,
    "absent": 2,
    "unknown": 3,
    None: 3
}

pramana_map = {
    "perception": 0,
    "non_apprehension": 1,
    "comparison_analogy": 2,
    "testimony": 3,
    "inference": 4,
    "postulation_derivation": 5
}

reasoning_strategy_map = {
    None: 0,
    "direct_observation": 1,
    "absence_detection": 2,
    "spatial_reasoning": 3,
    "written_evidence": 4,
    "numerical_comparison": 5,
    "multi_source_synthesis": 6,
    "action_testing": 7
}

id_to_verdict = {v: k for k, v in verdict_map.items()}
id_to_stance = {v: k for k, v in stance_map.items()}
id_to_pramana = {v: k for k, v in pramana_map.items()}

### 6: Parse one record into model-ready format

In [125]:
def parse_record(record):
    claim = record["claim"]

    evidence_texts = []
    evidence_stances = []
    evidence_triples = []

    for ev in record["evidence"]:
        text = ev["text"] if ev["text"] is not None else ""
        evidence_texts.append(text)

        stance_id = stance_map.get(ev.get("stance"), stance_map["unknown"])
        evidence_stances.append(stance_id)

        triples = ev["triples"] if ev["triples"] is not None else []
        evidence_triples.extend(triples)

    verdict_id = verdict_map[record["verdict"]["label"]]

    pramana_id = pramana_map[record["epistemic"]["pramana_primary"]]
    pramana_weight = float(record["epistemic"]["confidence_weight"])

    reasoning = record.get("reasoning")
    if reasoning is None:
        reasoning_strategy = None
    else:
        reasoning_strategy = reasoning.get("strategy")

    reasoning_id = reasoning_strategy_map[reasoning_strategy]

    claim_triples = record["claim_triples"] if record["claim_triples"] is not None else []

    return {
        "id": record["id"],
        "claim": claim,
        "evidence_texts": evidence_texts,
        "evidence_stances": evidence_stances,
        "claim_triples": claim_triples,
        "evidence_triples": evidence_triples,
        "verdict": verdict_id,
        "pramana": pramana_id,
        "pramana_weight": pramana_weight,
        "reasoning_id": reasoning_id,
        "dataset": record["provenance"]["dataset"]
    }

In [126]:
parsed = parse_record(data[0])

for key, value in parsed.items():
    print(key, ":", value)

id : averitec-train-000001
claim : Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.
evidence_texts : ['Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014 No', 'Did Hunter Biden have any experience in Ukraine at the time he joined the board of the  Burisma energy company in 2014 No']
evidence_stances : [0, 0]
claim_triples : []
evidence_triples : []
verdict : 0
pramana : 3
pramana_weight : 0.8
reasoning_id : 0
dataset : averitec


### Notebook 2: Build PyTorch Dataset + Sentence Embeddings

In [127]:
import torch
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

### 1. Load sentence-transformer

In [128]:
device = "cuda" if torch.cuda.is_available() else "cpu"

text_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

TEXT_DIM = 384

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2262.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 2. Convert evidence list into one text block

For now, combine all evidence text into one string.

In [129]:
def combine_evidence_texts(evidence_texts):
    if len(evidence_texts) == 0:
        return ""

    return " ".join([
        text for text in evidence_texts
        if text is not None and len(text.strip()) > 0
    ])

### 3. Encode all records once

In [130]:
parsed_data = [parse_record(record) for record in data ]

claims = []
evidences = []

for item in parsed_data:
    claims.append(item["claim"])
    evidences.append(combine_evidence_texts(item["evidence_texts"]))

print("Total parsed records:", len(parsed_data))
print("Records filtered out:", len(data) - len(parsed_data))
print("Example claim:", claims[0])
print("Example evidence:", evidences[0])

Total parsed records: 5135
Records filtered out: 0
Example claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.
Example evidence: Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014 No Did Hunter Biden have any experience in Ukraine at the time he joined the board of the  Burisma energy company in 2014 No


In [131]:
from collections import defaultdict, Counter

stance_to_verdict = defaultdict(Counter)

for record in data:
    verdict = record["verdict"]["label"]

    for ev in record["evidence"]:
        stance = ev.get("stance")
        if stance is None:
            stance = "unknown"

        stance_to_verdict[stance][verdict] += 1

for stance, counts in stance_to_verdict.items():
    print("\nStance:", stance)
    print(counts)

    total = sum(counts.values())
    majority = counts.most_common(1)[0]

    print("Majority verdict:", majority[0])
    print("Majority accuracy:", majority[1] / total)


Stance: supports
Counter({'supported': 2710})
Majority verdict: supported
Majority accuracy: 1.0

Stance: refutes
Counter({'refuted': 6357})
Majority verdict: refuted
Majority accuracy: 1.0

Stance: unknown
Counter({'not_enough_evidence': 936})
Majority verdict: not_enough_evidence
Majority accuracy: 1.0

Stance: absent
Counter({'supported': 750})
Majority verdict: supported
Majority accuracy: 1.0


### 4. Create embeddings

In [132]:
claim_embeddings = text_model.encode(
    claims,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

evidence_embeddings = text_model.encode(
    evidences,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

print(claim_embeddings.shape)
print(evidence_embeddings.shape)

Batches: 100%|██████████| 161/161 [00:13<00:00, 11.59it/s]

torch.Size([5135, 384])
torch.Size([5135, 384])


### 5. Create Dataset class

In [133]:
class EpistemicClaimDataset(Dataset):
    def __init__(self, parsed_data, claim_embeddings, evidence_embeddings):
        self.data = parsed_data
        self.claim_embeddings = claim_embeddings
        self.evidence_embeddings = evidence_embeddings

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        return {
            "id": item["id"],

            "claim_emb": self.claim_embeddings[idx].float(),
            "evidence_emb": self.evidence_embeddings[idx].float(),

            "verdict": torch.tensor(item["verdict"], dtype=torch.long),

            # gold stance is label only, not input
            "stance": torch.tensor(
                item["evidence_stances"][0],
                dtype=torch.long
            ),

            "pramana": torch.tensor(item["pramana"], dtype=torch.long),

            "pramana_weight": torch.tensor(
                item["pramana_weight"],
                dtype=torch.float
            ),

            "reasoning_id": torch.tensor(
                item["reasoning_id"],
                dtype=torch.long
            ),

            "has_claim_triples": torch.tensor(
                len(item["claim_triples"]) > 0,
                dtype=torch.float
            ),

            "has_evidence_triples": torch.tensor(
                len(item["evidence_triples"]) > 0,
                dtype=torch.float
            )
        }

### 6. Create train/dev split

In [134]:
from sklearn.model_selection import train_test_split

indices = list(range(len(parsed_data)))

train_idx, dev_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=[item["verdict"] for item in parsed_data]
)

### 7. Create subset helper

In [135]:
def subset_list(items, indices):
    return [items[i] for i in indices]

train_dataset = EpistemicClaimDataset(
    subset_list(parsed_data, train_idx),
    claim_embeddings[train_idx],
    evidence_embeddings[train_idx]
)

dev_dataset = EpistemicClaimDataset(
    subset_list(parsed_data, dev_idx),
    claim_embeddings[dev_idx],
    evidence_embeddings[dev_idx]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False
)

### 8. Test one batch

In [136]:
batch = next(iter(train_loader))

for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(key, value.shape, value.dtype)
    else:
        print(key, type(value), value[:2])

id <class 'list'> ['averitec-train-002511', 'claim-FloorPlan203-absence-ref-000110']
claim_emb torch.Size([32, 384]) torch.float32
evidence_emb torch.Size([32, 384]) torch.float32
verdict torch.Size([32]) torch.int64
stance torch.Size([32]) torch.int64
pramana torch.Size([32]) torch.int64
pramana_weight torch.Size([32]) torch.float32
reasoning_id torch.Size([32]) torch.int64
has_claim_triples torch.Size([32]) torch.float32
has_evidence_triples torch.Size([32]) torch.float32


### Notebook 3: Build the Batched Epistemic Model

This version supports batch training.

In [137]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### 2. Stance Predictor

This predicts stance from claim and evidence embeddings.

In [138]:
class StancePredictor(nn.Module):
    def __init__(self, dim=384, hidden_dim=256, num_stances=4):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_stances)
        )

    def forward(self, claim_emb, evidence_emb):
        pair = torch.cat(
            [
                claim_emb,
                evidence_emb,
                torch.abs(claim_emb - evidence_emb),
                claim_emb * evidence_emb
            ],
            dim=-1
        )

        return self.mlp(pair)

### 3. Pramana-aware Fusion Layer

This is a simple GNN-like reasoning layer.

In [139]:
class PramanaFusionLayer(nn.Module):
    def __init__(self, dim=384):
        super().__init__()

        self.claim_update = nn.Linear(dim * 4, dim)
        self.evidence_update = nn.Linear(dim * 4, dim)

        self.norm_claim = nn.LayerNorm(dim)
        self.norm_evidence = nn.LayerNorm(dim)

    def forward(
        self,
        claim_h,
        evidence_h,
        stance_h,
        pramana_h,
        reasoning_h,
        pramana_weight
    ):
        weight = pramana_weight.unsqueeze(-1)

        epistemic_h = (
            stance_h
            + pramana_h * weight
            + reasoning_h * weight
        ) / 3.0

        claim_input = torch.cat(
            [claim_h, evidence_h, stance_h, epistemic_h],
            dim=-1
        )

        evidence_input = torch.cat(
            [evidence_h, claim_h, pramana_h, reasoning_h],
            dim=-1
        )

        claim_new = self.claim_update(claim_input)
        evidence_new = self.evidence_update(evidence_input)

        claim_h = self.norm_claim(claim_h + F.relu(claim_new))
        evidence_h = self.norm_evidence(evidence_h + F.relu(evidence_new))

        return claim_h, evidence_h, epistemic_h

### 4. Full Model

In [140]:
class EpistemicVerificationModel(nn.Module):
    def __init__(
        self,
        text_dim=384,
        hidden_dim=384,
        num_verdicts=4,
        num_stances=4,
        num_pramanas=6,
        num_reasoning=8
    ):
        super().__init__()

        self.claim_proj = nn.Linear(text_dim, hidden_dim)
        self.evidence_proj = nn.Linear(text_dim, hidden_dim)

        self.stance_predictor = StancePredictor(
            dim=hidden_dim,
            hidden_dim=256,
            num_stances=num_stances
        )

        self.stance_emb = nn.Linear(num_stances, hidden_dim)

        self.pramana_emb = nn.Embedding(num_pramanas, hidden_dim)
        self.null_reasoning = nn.Parameter(torch.randn(1, hidden_dim))

        self.fusion1 = PramanaFusionLayer(hidden_dim)
        self.fusion2 = PramanaFusionLayer(hidden_dim)

        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_verdicts)
        )

        self.reasoning_head = nn.Linear(hidden_dim, num_reasoning)
        self.null_reasoning = nn.Parameter(torch.randn(1, hidden_dim))

    def forward(self, batch):
        claim_emb = batch["claim_emb"]
        evidence_emb = batch["evidence_emb"]

        claim_h = self.claim_proj(claim_emb)
        evidence_h = self.evidence_proj(evidence_emb)

        # Predict stance internally
        stance_logits = self.stance_predictor(claim_h, evidence_h)
        stance_probs = F.softmax(stance_logits, dim=-1)
        stance_h = self.stance_emb(stance_probs)

        pramana_h = self.pramana_emb(batch["pramana"])
        reasoning_h = self.null_reasoning.expand(claim_h.size(0), -1)

        pramana_weight = batch["pramana_weight"]

        claim_h, evidence_h, epistemic_h = self.fusion1(
            claim_h,
            evidence_h,
            stance_h,
            pramana_h,
            reasoning_h,
            pramana_weight
        )

        claim_h, evidence_h, epistemic_h = self.fusion2(
            claim_h,
            evidence_h,
            stance_h,
            pramana_h,
            reasoning_h,
            pramana_weight
        )

        triple_flags = torch.stack(
            [
                batch["has_claim_triples"],
                batch["has_evidence_triples"]
            ],
            dim=-1
        )

        final_h = torch.cat(
            [
                claim_h,
                evidence_h,
                stance_h,
                epistemic_h,
                triple_flags
            ],
            dim=-1
        )

        verdict_logits = self.verdict_head(final_h)
        reasoning_logits = self.reasoning_head(claim_h)

        return {
            "verdict_logits": verdict_logits,
            "stance_logits": stance_logits,
            "reasoning_logits": reasoning_logits
        }

### 5. Create model

In [141]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = EpistemicVerificationModel(
    text_dim=384,
    hidden_dim=384,
    num_verdicts=4,
    num_stances=4,
    num_pramanas=6,
    num_reasoning=8
).to(device)

print(model)

EpistemicVerificationModel(
  (claim_proj): Linear(in_features=384, out_features=384, bias=True)
  (evidence_proj): Linear(in_features=384, out_features=384, bias=True)
  (stance_predictor): StancePredictor(
    (mlp): Sequential(
      (0): Linear(in_features=1536, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=256, out_features=4, bias=True)
    )
  )
  (stance_emb): Linear(in_features=4, out_features=384, bias=True)
  (pramana_emb): Embedding(6, 384)
  (fusion1): PramanaFusionLayer(
    (claim_update): Linear(in_features=1536, out_features=384, bias=True)
    (evidence_update): Linear(in_features=1536, out_features=384, bias=True)
    (norm_claim): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (norm_evidence): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  )
  (fusion2): PramanaFusionLayer(
    (claim_update): Linear(in_features=1536, out_features=384, bias=True)
    (evidence_update): Linear

### 6. Test forward pass

In [142]:
batch = next(iter(train_loader))

batch = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in batch.items()
}

outputs = model(batch)

print(outputs["verdict_logits"].shape)
print(outputs["stance_logits"].shape)
print(outputs["reasoning_logits"].shape)

torch.Size([32, 4])
torch.Size([32, 4])
torch.Size([32, 8])


In [143]:
batch["pramana"]
batch["reasoning_id"]

tensor([1, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2,
        0, 0, 1, 2, 0, 0, 0, 0])

### Notebook 4: Training Loop with Multi-Task Loss

Now we train the model with:

verdict loss  +  stance loss  +  reasoning loss

Remember:

stance is target only, not input

### 1. Optimizer and losses

In [144]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

criterion_verdict = nn.CrossEntropyLoss()
criterion_stance = nn.CrossEntropyLoss()
criterion_reasoning = nn.CrossEntropyLoss()

### 2. Move batch to device helper

In [145]:
def move_batch_to_device(batch, device):
    new_batch = {}

    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            new_batch[key] = value.to(device)
        else:
            new_batch[key] = value

    return new_batch

### 3. Training function

In [146]:
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    device,
    alpha_stance=0.5,
    beta_reasoning=0.3
):
    model.train()

    total_loss = 0.0
    total_verdict_loss = 0.0
    total_stance_loss = 0.0
    total_reasoning_loss = 0.0

    correct_verdict = 0
    correct_stance = 0
    total = 0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        outputs = model(batch)

        loss_verdict = criterion_verdict(
            outputs["verdict_logits"],
            batch["verdict"]
        )

        loss_stance = criterion_stance(
            outputs["stance_logits"],
            batch["stance"]
        )

        loss_reasoning = criterion_reasoning(
            outputs["reasoning_logits"],
            batch["reasoning_id"]
        )

        loss = (
            loss_verdict
            + alpha_stance * loss_stance
            + beta_reasoning * loss_reasoning
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = batch["verdict"].size(0)

        total_loss += loss.item() * batch_size
        total_verdict_loss += loss_verdict.item() * batch_size
        total_stance_loss += loss_stance.item() * batch_size
        total_reasoning_loss += loss_reasoning.item() * batch_size

        verdict_preds = outputs["verdict_logits"].argmax(dim=-1)
        stance_preds = outputs["stance_logits"].argmax(dim=-1)

        correct_verdict += (verdict_preds == batch["verdict"]).sum().item()
        correct_stance += (stance_preds == batch["stance"]).sum().item()
        total += batch_size

    return {
        "loss": total_loss / total,
        "verdict_loss": total_verdict_loss / total,
        "stance_loss": total_stance_loss / total,
        "reasoning_loss": total_reasoning_loss / total,
        "verdict_acc": correct_verdict / total,
        "stance_acc": correct_stance / total
    }

### 4. Evaluation function

In [147]:
@torch.no_grad()
def evaluate(
    model,
    dataloader,
    device,
    alpha_stance=0.5,
    beta_reasoning=0.3
):
    model.eval()

    total_loss = 0.0
    total_verdict_loss = 0.0
    total_stance_loss = 0.0
    total_reasoning_loss = 0.0

    correct_verdict = 0
    correct_stance = 0
    correct_reasoning = 0
    total = 0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        outputs = model(batch)

        loss_verdict = criterion_verdict(
            outputs["verdict_logits"],
            batch["verdict"]
        )

        loss_stance = criterion_stance(
            outputs["stance_logits"],
            batch["stance"]
        )

        loss_reasoning = criterion_reasoning(
            outputs["reasoning_logits"],
            batch["reasoning_id"]
        )

        loss = (
            loss_verdict
            + alpha_stance * loss_stance
            + beta_reasoning * loss_reasoning
        )

        batch_size = batch["verdict"].size(0)

        total_loss += loss.item() * batch_size
        total_verdict_loss += loss_verdict.item() * batch_size
        total_stance_loss += loss_stance.item() * batch_size
        total_reasoning_loss += loss_reasoning.item() * batch_size

        verdict_preds = outputs["verdict_logits"].argmax(dim=-1)
        stance_preds = outputs["stance_logits"].argmax(dim=-1)
        reasoning_preds = outputs["reasoning_logits"].argmax(dim=-1)

        correct_verdict += (verdict_preds == batch["verdict"]).sum().item()
        correct_stance += (stance_preds == batch["stance"]).sum().item()
        correct_reasoning += (reasoning_preds == batch["reasoning_id"]).sum().item()

        total += batch_size

    return {
        "loss": total_loss / total,
        "verdict_loss": total_verdict_loss / total,
        "stance_loss": total_stance_loss / total,
        "reasoning_loss": total_reasoning_loss / total,
        "verdict_acc": correct_verdict / total,
        "stance_acc": correct_stance / total,
        "reasoning_acc": correct_reasoning / total
    }

### 5. Run training

In [148]:
NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    dev_metrics = evaluate(
        model,
        dev_loader,
        device
    )

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    print(
        "Train | "
        f"loss: {train_metrics['loss']:.4f} | "
        f"verdict_acc: {train_metrics['verdict_acc']:.4f} | "
        f"stance_acc: {train_metrics['stance_acc']:.4f}"
    )

    print(
        "Dev   | "
        f"loss: {dev_metrics['loss']:.4f} | "
        f"verdict_acc: {dev_metrics['verdict_acc']:.4f} | "
        f"stance_acc: {dev_metrics['stance_acc']:.4f} | "
        f"reasoning_acc: {dev_metrics['reasoning_acc']:.4f}"
    )


Epoch 1/10
Train | loss: 1.1241 | verdict_acc: 0.7235 | stance_acc: 0.6529
Dev   | loss: 0.8802 | verdict_acc: 0.7459 | stance_acc: 0.7537 | reasoning_acc: 0.9971

Epoch 2/10
Train | loss: 0.8704 | verdict_acc: 0.7554 | stance_acc: 0.7607
Dev   | loss: 0.8318 | verdict_acc: 0.7624 | stance_acc: 0.7644 | reasoning_acc: 0.9971

Epoch 3/10
Train | loss: 0.8226 | verdict_acc: 0.7739 | stance_acc: 0.7734
Dev   | loss: 0.8215 | verdict_acc: 0.7605 | stance_acc: 0.7614 | reasoning_acc: 0.9971

Epoch 4/10
Train | loss: 0.7863 | verdict_acc: 0.7868 | stance_acc: 0.7860
Dev   | loss: 0.7939 | verdict_acc: 0.7790 | stance_acc: 0.7683 | reasoning_acc: 0.9981

Epoch 5/10
Train | loss: 0.7376 | verdict_acc: 0.8028 | stance_acc: 0.8033
Dev   | loss: 0.7969 | verdict_acc: 0.7868 | stance_acc: 0.7702 | reasoning_acc: 0.9981

Epoch 6/10
Train | loss: 0.6754 | verdict_acc: 0.8313 | stance_acc: 0.8262
Dev   | loss: 0.8483 | verdict_acc: 0.7585 | stance_acc: 0.7780 | reasoning_acc: 0.9981

Epoch 7/10
Trai

### 6. Save model

In [149]:
torch.save(
    model.state_dict(),
    "pa_ehgnn_model.pt"
)

### 7. Load model later

In [150]:
model = EpistemicVerificationModel(
    text_dim=384,
    hidden_dim=384,
    num_verdicts=4,
    num_stances=4,
    num_pramanas=6,
    num_reasoning=8
).to(device)

model.load_state_dict(
    torch.load("pa_ehgnn_model.pt", map_location=device)
)

model.eval()

EpistemicVerificationModel(
  (claim_proj): Linear(in_features=384, out_features=384, bias=True)
  (evidence_proj): Linear(in_features=384, out_features=384, bias=True)
  (stance_predictor): StancePredictor(
    (mlp): Sequential(
      (0): Linear(in_features=1536, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=256, out_features=4, bias=True)
    )
  )
  (stance_emb): Linear(in_features=4, out_features=384, bias=True)
  (pramana_emb): Embedding(6, 384)
  (fusion1): PramanaFusionLayer(
    (claim_update): Linear(in_features=1536, out_features=384, bias=True)
    (evidence_update): Linear(in_features=1536, out_features=384, bias=True)
    (norm_claim): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (norm_evidence): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  )
  (fusion2): PramanaFusionLayer(
    (claim_update): Linear(in_features=1536, out_features=384, bias=True)
    (evidence_update): Linear

### Notebook 5: Diagnostic Tests for Shortcut Learning
This notebook checks whether your model learned real epistemic reasoning or just easy patterns.

### 1. Basic prediction inspection

In [151]:
@torch.no_grad()
def inspect_predictions(model, dataloader, dataset, device, n=10):
    model.eval()

    shown = 0

    for batch_idx, batch in enumerate(dataloader):
        batch = move_batch_to_device(batch, device)
        outputs = model(batch)

        verdict_preds = outputs["verdict_logits"].argmax(dim=-1)
        stance_preds = outputs["stance_logits"].argmax(dim=-1)
        reasoning_preds = outputs["reasoning_logits"].argmax(dim=-1)

        for i in range(len(verdict_preds)):
            item_idx = batch_idx * dataloader.batch_size + i
            if item_idx >= len(dataset.data):
                continue

            item = dataset.data[item_idx]

            print("=" * 80)
            print("ID:", item["id"])
            print("Dataset:", item["dataset"])
            print("Claim:", item["claim"])
            print("Evidence:", combine_evidence_texts(item["evidence_texts"]))

            print("\nGold verdict:", id_to_verdict[item["verdict"]])
            print("Pred verdict:", id_to_verdict[verdict_preds[i].item()])

            print("\nGold stance:", id_to_stance[item["evidence_stances"][0]])
            print("Pred stance:", id_to_stance[stance_preds[i].item()])

            print("\nGold pramana:", id_to_pramana[item["pramana"]])
            print("Pramana weight:", item["pramana_weight"])

            print("Gold reasoning ID:", item["reasoning_id"])
            print("Pred reasoning ID:", reasoning_preds[i].item())

            shown += 1

            if shown >= n:
                return

In [152]:
inspect_predictions(
    model=model,
    dataloader=dev_loader,
    dataset=dev_dataset,
    device=device,
    n=5
)

ID: claim-FloorPlan15-absence-sup-000084
Dataset: ai2thor
Claim: There is no CD in this scene.
Evidence: No sensor evidence found for this object type.

Gold verdict: supported
Pred verdict: supported

Gold stance: absent
Pred stance: absent

Gold pramana: non_apprehension
Pramana weight: 0.75
Gold reasoning ID: 2
Pred reasoning ID: 2
ID: averitec-train-000855
Dataset: averitec
Claim: President Buhari of Nigeria has decided to resign but some Nigerians are against it.
Evidence: Where was this claim published? Igbowatch What does Igbowatch say about the claim now? APPEAL /Apology Over False Content.


 

An Earlier Version of this content was rated False by Dubawa via https://dubawa.org/buharis-would-be-resignation-plot-is-a-hoax/

THE version of the Incorrectly stated that President Muhammadu Buhari said he wanted to resign that Nigerians are saying No.


We have since punished the Content creator over such False News and wish to apologize to our Readers.

Gold verdict: refuted
Pred ve

### 2. Verdict-only baseline

This tells whether your dataset is too easy.

In [153]:
class StanceOnlyBaseline(nn.Module):
    def __init__(self, num_stances=4, num_verdicts=4):
        super().__init__()
        self.classifier = nn.Linear(num_stances, num_verdicts)

    def forward(self, stance_gold):
        stance_onehot = F.one_hot(
            stance_gold,
            num_classes=4
        ).float()

        return self.classifier(stance_onehot)

In [154]:
baseline = StanceOnlyBaseline().to(device)

baseline_optimizer = torch.optim.AdamW(
    baseline.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

In [155]:
def train_stance_only_baseline(model, dataloader, optimizer, device):
    model.train()

    total = 0
    correct = 0
    total_loss = 0.0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        logits = model(batch["stance"])

        loss = criterion(logits, batch["verdict"])
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=-1)

        correct += (preds == batch["verdict"]).sum().item()
        total += batch["verdict"].size(0)
        total_loss += loss.item() * batch["verdict"].size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

In [156]:
@torch.no_grad()
def eval_stance_only_baseline(model, dataloader, device):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        logits = model(batch["stance"])
        loss = criterion(logits, batch["verdict"])

        preds = logits.argmax(dim=-1)

        correct += (preds == batch["verdict"]).sum().item()
        total += batch["verdict"].size(0)
        total_loss += loss.item() * batch["verdict"].size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

In [157]:
for epoch in range(5):
    train_m = train_stance_only_baseline(
        baseline,
        train_loader,
        baseline_optimizer,
        device
    )

    dev_m = eval_stance_only_baseline(
        baseline,
        dev_loader,
        device
    )

    print(
        f"Epoch {epoch + 1} | "
        f"train acc: {train_m['acc']:.4f} | "
        f"dev acc: {dev_m['acc']:.4f}"
    )

Epoch 1 | train acc: 0.2899 | dev acc: 0.9377
Epoch 2 | train acc: 0.9384 | dev acc: 0.9377
Epoch 3 | train acc: 0.9384 | dev acc: 0.9377
Epoch 4 | train acc: 0.9460 | dev acc: 1.0000
Epoch 5 | train acc: 1.0000 | dev acc: 1.0000


### 3. Pramana ablation test

Remove pramana information at evaluation.

In [158]:
@torch.no_grad()
def evaluate_without_pramana(model, dataloader, device):
    model.eval()

    total = 0
    correct = 0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        # Replace pramana with fixed value
        batch["pramana"] = torch.zeros_like(batch["pramana"])

        # Replace pramana weight with neutral weight
        batch["pramana_weight"] = torch.ones_like(batch["pramana_weight"]) * 0.5

        outputs = model(batch)
        preds = outputs["verdict_logits"].argmax(dim=-1)

        correct += (preds == batch["verdict"]).sum().item()
        total += batch["verdict"].size(0)

    return correct / total

In [159]:
normal_metrics = evaluate(model, dev_loader, device)
no_pramana_acc = evaluate_without_pramana(model, dev_loader, device)

print("Normal dev accuracy:", normal_metrics["verdict_acc"])
print("Without pramana accuracy:", no_pramana_acc)

Normal dev accuracy: 0.7643622200584226
Without pramana accuracy: 0.7643622200584226


### 4. Triple ablation test

Remove triple flags.

In [160]:
@torch.no_grad()
def evaluate_without_triple_flags(model, dataloader, device):
    model.eval()

    total = 0
    correct = 0

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)

        batch["has_claim_triples"] = torch.zeros_like(batch["has_claim_triples"])
        batch["has_evidence_triples"] = torch.zeros_like(batch["has_evidence_triples"])

        outputs = model(batch)
        preds = outputs["verdict_logits"].argmax(dim=-1)

        correct += (preds == batch["verdict"]).sum().item()
        total += batch["verdict"].size(0)

    return correct / total

In [161]:
no_triple_acc = evaluate_without_triple_flags(
    model,
    dev_loader,
    device
)

print("Normal dev accuracy:", normal_metrics["verdict_acc"])
print("Without triple flags accuracy:", no_triple_acc)

Normal dev accuracy: 0.7643622200584226
Without triple flags accuracy: 0.7643622200584226


### 5. Dataset split evaluation

Evaluate AVeriTeC and AI2THOR separately.

In [162]:
def filter_dataset_by_source(dataset, source_name):
    indices = [
        i for i, item in enumerate(dataset.data)
        if item["dataset"] == source_name
    ]

    subset_data = [dataset.data[i] for i in indices]
    subset_claim_emb = dataset.claim_embeddings[indices]
    subset_evidence_emb = dataset.evidence_embeddings[indices]

    return EpistemicClaimDataset(
        subset_data,
        subset_claim_emb,
        subset_evidence_emb
    )

In [163]:
dev_averitec = filter_dataset_by_source(dev_dataset, "averitec")
dev_ai2thor = filter_dataset_by_source(dev_dataset, "ai2thor")

dev_averitec_loader = DataLoader(
    dev_averitec,
    batch_size=32,
    shuffle=False
)

dev_ai2thor_loader = DataLoader(
    dev_ai2thor,
    batch_size=32,
    shuffle=False
)

In [164]:
if len(dev_averitec) > 0:
    print("AVeriTeC:", evaluate(model, dev_averitec_loader, device))

if len(dev_ai2thor) > 0:
    print("AI2THOR:", evaluate(model, dev_ai2thor_loader, device))

AVeriTeC: {'loss': 1.7049853872924268, 'verdict_loss': 1.3300362737766633, 'stance_loss': 0.747430118813449, 'reasoning_loss': 0.004113631082305066, 'verdict_acc': 0.6508422664624809, 'stance_acc': 0.6875957120980092, 'reasoning_acc': 1.0}
AI2THOR: {'loss': 0.18691220513002121, 'verdict_loss': 0.1407310577896349, 'stance_loss': 0.08073055596753238, 'reasoning_loss': 0.019386213621433408, 'verdict_acc': 0.9625668449197861, 'stance_acc': 0.9625668449197861, 'reasoning_acc': 0.9946524064171123}


### 6. Counterfactual pramana test

Change only pramana type and weight.

In [165]:
@torch.no_grad()
def counterfactual_pramana_test(model, batch, device):
    model.eval()

    batch = move_batch_to_device(batch, device)

    pramana_names = list(pramana_map.keys())

    results = []

    for pramana_name, pramana_id in pramana_map.items():
        cf_batch = batch.copy()

        cf_batch["pramana"] = torch.ones_like(batch["pramana"]) * pramana_id

        if pramana_name == "perception":
            weight = 0.90
        elif pramana_name == "testimony":
            weight = 0.85
        elif pramana_name == "non_apprehension":
            weight = 0.80
        elif pramana_name == "comparison_analogy":
            weight = 0.75
        elif pramana_name == "inference":
            weight = 0.70
        else:
            weight = 0.60

        cf_batch["pramana_weight"] = torch.ones_like(batch["pramana_weight"]) * weight

        outputs = model(cf_batch)

        probs = F.softmax(outputs["verdict_logits"], dim=-1)

        results.append((pramana_name, probs.detach().cpu()))

    return results

In [166]:
batch = next(iter(dev_loader))
results = counterfactual_pramana_test(model, batch, device)

for pramana_name, probs in results:
    print("\nPramana:", pramana_name)
    print(probs[0])


Pramana: perception
tensor([9.8037e-01, 1.7234e-02, 2.3828e-03, 1.1802e-05])

Pramana: non_apprehension
tensor([9.9999e-01, 8.6175e-07, 5.0800e-06, 7.0778e-08])

Pramana: comparison_analogy
tensor([9.9010e-01, 1.5701e-03, 8.3229e-03, 1.0885e-05])

Pramana: testimony
tensor([9.7533e-01, 1.0370e-03, 2.3625e-02, 4.3735e-06])

Pramana: inference
tensor([9.8708e-01, 1.0772e-03, 1.1837e-02, 7.5429e-06])

Pramana: postulation_derivation
tensor([9.9861e-01, 2.6395e-04, 1.1218e-03, 2.6196e-06])
